In [1]:
!pip install pandas numpy scikit-learn

In [ ]:
#import all library

import pandas as pd
import numpy as np
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('✅ All libraries imported successfully!')

✅ All libraries imported successfully!


In [3]:
# Load the dataset
df = pd.read_csv('Crop_recommendation.csv')

print(f'Shape     : {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Columns   : {list(df.columns)}')
print(f'Crop types: {df["label"].nunique()} unique classes')
print(f'Missing   : {df.isnull().sum().sum()} null values')
print()
df.head()

Shape     : 2200 rows × 8 columns
Columns   : ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label']
Crop types: 22 unique classes
Missing   : 0 null values



,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


In [4]:
# Check class distribution
print('Crop distribution:')
print(df['label'].value_counts())

Crop distribution:
label
rice           100
maize          100
chickpea       100
kidneybeans    100
pigeonpeas     100
mothbeans      100
mungbean       100
blackgram      100
lentil         100
pomegranate    100
banana         100
mango          100
grapes         100
watermelon     100
muskmelon      100
apple          100
orange         100
papaya         100
coconut        100
cotton         100
jute           100
coffee         100
Name: count, dtype: int64


In [ ]:
#setting features and target column
FEATURE_COLUMNS = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
TARGET_COLUMN   = 'label'

# Drop duplicates & nulls
df = df.drop_duplicates().dropna()

X = df[FEATURE_COLUMNS].values
y = df[TARGET_COLUMN].values

# Encode labels
le = LabelEncoder()
y_enc = le.fit_transform(y)
print(f'Classes : {list(le.classes_)}')

# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# Feature scaling
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'\nTrain samples : {len(X_train)}')
print(f'Test  samples : {len(X_test)}')
print('✅ Preprocessing done!')

Classes : ['apple', 'banana', 'blackgram', 'chickpea', 'coconut', 'coffee', 'cotton', 'grapes', 'jute', 'kidneybeans', 'lentil', 'maize', 'mango', 'mothbeans', 'mungbean', 'muskmelon', 'orange', 'papaya', 'pigeonpeas', 'pomegranate', 'rice', 'watermelon']

Train samples : 1760
Test  samples : 440
✅ Preprocessing done!


In [ ]:
#Making model and training it 

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Cross validation
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
print(f'Cross-val Accuracy : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print('✅ Model trained!')

Cross-val Accuracy : 0.9943 ± 0.0054
✅ Model trained!


In [ ]:
#Predicting and testing the model
y_pred = model.predict(X_test)
acc    = accuracy_score(y_test, y_pred)

print(f'Test Accuracy : {acc * 100:.2f}%\n')
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=le.classes_))

Test Accuracy : 99.55%

Classification Report:
              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      1.00      1.00        20
   blackgram       1.00      0.95      0.97        20
    chickpea       1.00      1.00      1.00        20
     coconut       1.00      1.00      1.00        20
      coffee       1.00      1.00      1.00        20
      cotton       1.00      1.00      1.00        20
      grapes       1.00      1.00      1.00        20
        jute       0.95      1.00      0.98        20
 kidneybeans       1.00      1.00      1.00        20
      lentil       1.00      1.00      1.00        20
       maize       0.95      1.00      0.98        20
       mango       1.00      1.00      1.00        20
   mothbeans       1.00      1.00      1.00        20
    mungbean       1.00      1.00      1.00        20
   muskmelon       1.00      1.00      1.00        20
      orange       1.00      1.00 

In [12]:
# Feature importance --> most important features for crop recommendation
print('Feature Importance:')
importances = model.feature_importances_
for feat, imp in sorted(zip(FEATURE_COLUMNS, importances), key=lambda x: -x[1]):
    bar = '█' * int(imp * 40)
    print(f'  {feat:12s} | {bar:<40s} {imp:.4f}')

Feature Importance:
  rainfall     | ████████                                 0.2196
  humidity     | ████████                                 0.2171
  K            | ███████                                  0.1808
  P            | ██████                                   0.1513
  N            | ████                                     0.1034
  temperature  | ███                                      0.0755
  ph           | ██                                       0.0523


In [ ]:
#making pickle file for model,scaler and label encoder
# Save model
with open('crop_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print('✅ Model saved   → crop_model.pkl')

# Save scaler
with open('crop_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('✅ Scaler saved  → crop_scaler.pkl')

# Save label encoder
with open('crop_label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
print('✅ Encoder saved → crop_label_encoder.pkl')

✅ Model saved   → crop_model.pkl
✅ Scaler saved  → crop_scaler.pkl
✅ Encoder saved → crop_label_encoder.pkl


In [10]:
# Load model back from disk
with open('crop_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)
print('✅ Model loaded   ← crop_model.pkl')

with open('crop_scaler.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)
print('✅ Scaler loaded  ← crop_scaler.pkl')

with open('crop_label_encoder.pkl', 'rb') as f:
    loaded_le = pickle.load(f)
print('✅ Encoder loaded ← crop_label_encoder.pkl')

✅ Model loaded   ← crop_model.pkl
✅ Scaler loaded  ← crop_scaler.pkl
✅ Encoder loaded ← crop_label_encoder.pkl


In [ ]:
# Giving new values
# [N, P, K, temperature, humidity, ph, rainfall]
sample_features = [117, 32, 34, 26.2724184, 52.12739421, 6.758792552, 127.1752928]

# Predict
sample        = np.array(sample_features).reshape(1, -1)
sample_scaled = loaded_scaler.transform(sample)
pred_index    = loaded_model.predict(sample_scaled)[0]
pred_proba    = loaded_model.predict_proba(sample_scaled)[0]
confidence    = pred_proba.max() * 100
crop_name     = loaded_le.inverse_transform([pred_index])[0]

print(f'Input   : {dict(zip(FEATURE_COLUMNS, sample_features))}')
print(f'\n🌾 Predicted Crop : {crop_name.upper()}')
print(f'📊 Confidence     : {confidence:.2f}%')

# Top 3
top3_idx   = pred_proba.argsort()[::-1][:3]
top3_crops = loaded_le.inverse_transform(top3_idx)
print('\n🏆 Top 3 Candidates:')
for crop, prob in zip(top3_crops, pred_proba[top3_idx]):
    print(f'   {crop:15s} : {prob * 100:.2f}%')

Input   : {'N': 117, 'P': 32, 'K': 34, 'temperature': 26.2724184, 'humidity': 52.12739421, 'ph': 6.758792552, 'rainfall': 127.1752928}

🌾 Predicted Crop : COFFEE
📊 Confidence     : 100.00%

🏆 Top 3 Candidates:
   coffee          : 100.00%
   watermelon      : 0.00%
   pomegranate     : 0.00%
